# Lineborn Sales LoRA
Fine-tunes Qwen3-4B-Instruct-2507 for consultative outbound sales. The frozen release benchmark is not used for training. Use a GPU runtime.

In [ ]:
!nvidia-smi
!rm -rf /content/lineborn-runtime
!git clone --depth 1 --branch lineborn-sales-lora https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git /content/lineborn-runtime
%cd /content/lineborn-runtime
!pip -q install -r training/requirements-colab.txt

In [ ]:
!python training/build_sales_corpus.py
!python training/validate_sales_corpus.py

In [ ]:
# Stage 1: assistant-only QLoRA supervised fine-tuning
!python training/train_sales_lora.py --max-length 1536 --epochs 2 --grad-accum 16

In [ ]:
# Stage 2: preference optimization. If a very small GPU OOMs here, keep the SFT adapter and benchmark it first.
!python training/train_sales_dpo.py --max-length 1536 --epochs 1 --grad-accum 16

In [ ]:
# Package adapters and metrics. The merged model is intentionally not created until the adapter passes the strict holdout benchmark.
!cd training/output && zip -qr /content/lineborn-sales-adapters.zip lineborn-sales-sft lineborn-sales-dpo
from google.colab import files
files.download('/content/lineborn-sales-adapters.zip')